# Tiền xử lý dữ liệu

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df_train = pd.read_csv('D:/ai_pratice_prj/Team9_ML_edu/House Price/data/txt/train.csv')
df_test = pd.read_csv('D:/ai_pratice_prj/Team9_ML_edu/House Price/data/txt/test.csv')


In [19]:
# Lưu Id của test set và SalePrice của train set
test_ids = df_test['Id']
train_ids = df_train['Id']

# Tách biến mục tiêu (SalePrice) và log-transform
# np.log1p(x) = log(1+x), an toàn khi x=0
y_train_log = np.log1p(df_train['SalePrice'])
df_train = df_train.drop(['Id', 'SalePrice'], axis=1)
df_test = df_test.drop('Id', axis=1)

# Gộp train và test để xử lý đồng nhất
df_all = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
print(f"Kích thước dữ liệu gộp (trước xử lý): {df_all.shape}")


Kích thước dữ liệu gộp (trước xử lý): (2919, 79)


## 1. Missing Value Imputation

In [20]:
# --- A. Nhóm 1: Biến phân loại có ý nghĩa 'None' khi NA ---
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'
]
for col in none_cols:
    if col in df_all.columns:
        df_all[col] = df_all[col].fillna('None')

# --- B. Nhóm 2: Biến định lượng có thể thay NA bằng 0 ---
zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 
    'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath'
]
for col in zero_cols:
    if col in df_all.columns:
        df_all[col] = df_all[col].fillna(0)

# --- C. Nhóm 3: Điền theo nhóm (LotFrontage theo Neighborhood) ---
if 'LotFrontage' in df_all.columns and 'Neighborhood' in df_all.columns:
    df_all['LotFrontage'] = df_all.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )

# --- D. Nhóm 4: Các cột phân loại còn lại dùng mode ---
for col in df_all.select_dtypes(include='object').columns:
    df_all[col] = df_all[col].fillna(df_all[col].mode()[0])

# --- E. Kiểm tra missing value từng cột ---
missing_cols = df_all.isnull().sum()[df_all.isnull().sum() > 0]
if len(missing_cols) > 0:
    print("Còn giá trị thiếu ở các cột sau:", missing_cols)
else:
    print("✅ Không còn giá trị thiếu.")


Còn giá trị thiếu ở các cột sau: MasVnrArea    23
dtype: int64


## 2. Feature Engineering

In [21]:
# --- Tổng diện tích sàn sử dụng ---
df_all['TotalSF'] = df_all['1stFlrSF'] + df_all['2ndFlrSF'] + df_all['TotalBsmtSF']

# --- Tuổi nhà tại thời điểm bán ---
df_all['HouseAge'] = df_all['YrSold'] - df_all['YearBuilt']
# --- Tuổi kể từ lần cải tạo gần nhất ---
df_all['RemodAge'] = df_all['YrSold'] - df_all['YearRemodAdd']

# --- Tổng số phòng (bao gồm phòng tắm) ---
df_all['TotalRooms'] = df_all['TotRmsAbvGrd'] + df_all['FullBath'] + df_all['HalfBath']

# --- Đặc trưng tương tác giữa chất lượng và diện tích ---
df_all['Qual_Area'] = df_all['OverallQual'] * df_all['GrLivArea']

# --- Tỷ lệ diện tích đất/sàn ---
df_all['LotRatio'] = df_all['LotArea'] / (df_all['GrLivArea'] + 1)

# --- Có garage hay không ---
df_all['HasGarage'] = (df_all['GarageArea'] > 0).astype(int)

# --- Có tầng hầm hay không ---
df_all['HasBsmt'] = (df_all['TotalBsmtSF'] > 0).astype(int)

# --- Có hồ bơi hay không ---
df_all['HasPool'] = (df_all['PoolArea'] > 0).astype(int)

# Kiểm tra nhanh các feature mới
df_all[['TotalSF', 'HouseAge', 'RemodAge', 'Qual_Area', 'LotRatio', 'HasGarage', 'HasBsmt', 'HasPool']].head()


,TotalSF,HouseAge,RemodAge,Qual_Area,LotRatio,HasGarage,HasBsmt,HasPool
0,2566.0,5,5,11970,4.938632,1,1,0
1,2524.0,31,31,7572,7.600950,1,1,0
2,2706.0,7,6,12502,6.295467,1,1,0
3,2473.0,91,36,12019,5.558789,1,1,0
4,3343.0,8,8,17584,6.484766,1,1,0


## 3. Normalization

In [22]:
# --- F. Mã hóa ordinal cho các biến chất lượng ---
quality_map = {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0}
qual_cols = ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC', 'GarageQual', 'FireplaceQu']
for col in qual_cols:
    if col in df_all.columns:
        df_all[col] = df_all[col].map(quality_map)

# --- G. One-hot encoding cho biến phân loại còn lại ---
cat_cols = df_all.select_dtypes(include='object').columns
df_all = pd.get_dummies(df_all, columns=cat_cols, drop_first=True)

# --- H. Log-transform các biến skewed numeric (tùy chỉnh nếu cần) ---
skewed_cols = ['GrLivArea', 'LotArea', '1stFlrSF', 'TotalBsmtSF']  # ví dụ
for col in skewed_cols:
    if col in df_all.columns:
        df_all[col] = np.log1p(df_all[col])

# --- I. Chuẩn hóa các biến số ---
num_cols = df_all.select_dtypes(include=[np.number]).columns.drop('SalePrice', errors='ignore')
scaler = StandardScaler()
df_all[num_cols] = scaler.fit_transform(df_all[num_cols])

# --- J. Log-transform target nếu cần ---
if 'SalePrice' in df_all.columns:
    df_all['SalePrice'] = np.log1p(df_all['SalePrice'])

print("✅ Hoàn tất chuẩn hóa, mã hóa & log-transform.")
print(f"Tổng số cột sau khi encoding: {df_all.shape[1]}")

# --- K. Lưu bản copy đã xử lý ---
df_all_processed = df_all.copy()


✅ Hoàn tất chuẩn hóa, mã hóa & log-transform.
Tổng số cột sau khi encoding: 249


## 4. Tách dữ liệu và xuất kết quả

In [23]:
# Tách lại train và test
X_train = df_all_processed.iloc[:len(y_train_log)]
X_test = df_all_processed.iloc[:len(y_train_log):]

# 4.1. Scaling (Chuẩn hóa)
# Chỉ lấy các cột số (loại trừ các cột one-hot đã là 0/1)
numerical_cols = df_train.select_dtypes(include=np.number).columns
# Lọc ra các cột số có trong X_train (vì một số có thể đã bị map)
cols_to_scale = [col for col in numerical_cols if col in X_train.columns]

scaler = StandardScaler()

# CHỈ fit trên X_train (để tránh rò rỉ dữ liệu từ test set)
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])

# DÙNG scaler đã fit để transform X_test
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])


C:\Users\Akama Asumi\AppData\Local\Temp\ipykernel_38100\3904301107.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
C:\Users\Akama Asumi\AppData\Local\Temp\ipykernel_38100\3904301107.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])


In [24]:
X_train.to_csv('D:/ai_pratice_prj/Team9_ML_edu/House Price/data/txt-preprocessing/train_processed.csv', index=False)
X_test.to_csv('D:/ai_pratice_prj/Team9_ML_edu/House Price/data/txt-preprocessing/test_processed.csv', index=False)
y_train_log.to_csv('D:/ai_pratice_prj/Team9_ML_edu/House Price/data/txt-preprocessing/y_train_log.csv', index=False, header=['SalePrice_Log'])

print("\n--- HOÀN THÀNH TIỀN XỬ LÝ ---")
print("Đã lưu 3 files:")
print("1. train_processed.csv (Dữ liệu train đã xử lý)")
print("2. test_processed.csv (Dữ liệu test đã xử lý)")
print("3. y_train_log.csv (Biến mục tiêu đã log-transform)")



--- HOÀN THÀNH TIỀN XỬ LÝ ---
Đã lưu 3 files:
1. train_processed.csv (Dữ liệu train đã xử lý)
2. test_processed.csv (Dữ liệu test đã xử lý)
3. y_train_log.csv (Biến mục tiêu đã log-transform)
